# 89 · Positional programs along the PT against a cluster-level view (CellSpectra)

A cluster-level analysis such as CellSpectra (Klötzer et al. 2025, Nat Genet; PMID 40775269) sees each PT segment as
one number per gene. The elastic curve sees each gene as a profile. This notebook asks what that adds: first in
theory and on synthetic programs written into the real design (can the curve show it?), then in the data
(boundary programs, positional programs, CellSpectra's own programs, and CellSpectra's PT disease features in the IR
mice).

**Protocol.** `docs/results/pt-positional-programs.md` (frozen before any fit, commit 35afb77).

| Part | Question |
|---|---|
| T | Can the curve show boundary programs, gradients and positions, given its noise? (semi-synthetic) |
| H1 | Are there boundary-centred pathway programs beyond chance? (primary) |
| E1 | Do pathways group into a few positional programs, and do they replicate between mice? |
| E2 | What does the cluster view say about the same pathways? |
| E3 | CellSpectra's species lists, conserved functions and disease features along the healthy PT |
| E4 | CellSpectra's PT disease features in the IR mice: where they change, and where coordination is lost |

**What bounds the reading.** Two healthy and two IR male mice, one male human donor (two cortex sections). The
coordinate's fold noise exceeds the within-segment spread in mouse S1 and S2 (C14). Human positions are descriptive.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import zlib
import json
import os
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

PKG = results_root / 'package_pseudospace'
EXPORT = {'hvm': PKG / 'hvm' / 'pt_units.h5ad', 'aki': PKG / 'aki' / 'pt_units.h5ad'}
STAGE01 = PKG / 'runs' / 'hvm' / '01_units'
RUN_HVM = PKG / 'runs' / 'hvm'
upstream13 = results_root / 'minimal_pt_scfates'
T71 = results_root / 'pt_pathways_package_pseudospace'
T72 = results_root / 'pt_package_coordinate_validation'
T81 = results_root / 'pt_published_findings'
T84 = results_root / 'pt_within_segment_pathways'
T86 = results_root / 'pt_non_one_to_one_package'
EXT = data_root / 'external'
CS = EXT / 'klotzer2025_cellspectra'
QGO = EXT / 'quickgo_2026-10-14' / 'GO_0002931.tsv'
LIBS = data_root / 'mouse_vs_human' / 'pathway_gene_sets'
FROZEN = project / 'docs' / 'results' / 'positional-programs'
KNOWN = project / 'docs' / 'results' / 'known-programs'
output = results_root / 'pt_positional_programs'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

DS = 'human_vs_mouse'
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
SPECIES = {'mouse': MOUSE, 'human': HUMAN}
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
AKI4 = CONTROLS + IRS
REQUIRED = [EXPORT['hvm'], EXPORT['aki'], *[STAGE01 / f'{s}_units.h5ad' for s in MOUSE + HUMAN],
            upstream13 / 'ortholog_map_used.csv', T72 / 'tables' / 'transitions.csv',
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json')],
            T81 / 'tables' / 'programs.csv', T84 / 'tables' / 'segment_pathways.csv',
            KNOWN / 'programs_hvm_members.csv', FROZEN / 'cellspectra_features.csv',
            CS / 'supplementary_table_5.csv', CS / 'supplementary_table_8.csv', QGO,
            *[LIBS / f'{lib}.json' for lib in ('GO_Biological_Process_2023', 'KEGG_2019_Mouse')],
            RUN_HVM / '03_integration' / 'pass1_labelled.h5ad', RUN_HVM / '03_integration' / 'pass2.h5ad']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70–72, 81, 84 and 86 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '89.positional_programs.1'
FINGERPRINT = {'hvm': 'efafc9ac2e99', 'aki': 'bb4a80c1b484'}
FROZEN_SHA = {'cellspectra_features.csv': None, 'GO_0002931.tsv': 'aacfb82369a90f3a9c99527725b3de1cfa4ad8e5e9325f0831a8d642bdcfe451',
              'supplementary_table_5.csv': '77f3cb3059ed9a747a414bdfd6aaa0eb7ffa1bdf6b7ce6039ee537c2d7e5adaa',
              'supplementary_table_8.csv': 'a73f8783c38f40c54a94c664a03e37cfc6c597f706a4c3e3a322bac76a46d0ef'}
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))
SEED = 89
# Protocol constants
DETECTION, USABLE = .02, .05            # PT-expressed (86) and usable genes
EFFECT = .25
MIN_SET, MIN_FEATURE = 10, 5
N_PERM, N_PERM_SENS = 1000, 200
N_SIM, SIM_GENES = 100, 15
AMPLITUDES = (.5, 1., 2.)
ARI_MIN, R_ELIGIBLE, R_NEAREST = .4, .5, .5
COORD_WIDTH, N_RANDOM = .25, 100
SEGMENTS = ('S1', 'S2', 'S3')
LABEL_ORIENTATION = ['Cyp2e1', 'Cyp7b1', 'Gatm', 'Slc7a13', 'Slc13a3', 'Slc22a6', 'Slc5a2', 'Slc22a7', 'Slc5a12',
                     'Aqp2', 'Aqp4', 'Atp6v0d2', 'Rhcg', 'Slc14a2', 'Wnt7b']

MM = 1 / 25.4
INK, LIGHT, GREY = '#0b0b0b', '#d9d8d4', '#9a9893'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'HUK1_COR1': '#D55E00', 'HUK1_MED1': '#E69F00',
              'IR2A2': '#B2182B', 'IR2A4': '#EF8A62'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 200)
GUARDS, READING = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

G1 `hvm_inputs` reproduces 71; G2 72's transitions; G3 the stage-01 native panels match the export unit by unit;
G4 frozen files and external data match their sha256; G5 86's cached W fits cover the PT-expressed mouse genes;
G6 the fold refits map onto ≥ 90% of mouse U_PT units; G7 the AKI control transitions are ordered.

In [ ]:
import anndata as ad
from scipy import sparse
from sklearn.metrics import adjusted_rand_score

from pseudospace import package_refits as refit_module
from pseudospace import positional_programs as pp
from pseudospace import published_findings as pf
from pseudospace import species_calls as sc
from pseudospace import transition_sharpness as ts
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import hvm_inputs, read_pt_units, support_positions
from pseudospace.pathways import build_pathway_membership
from pseudospace.program_localization import segment_pseudobulk
from pseudospace.stage_cache import cached_payload, digest

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
UNITS = read_pt_units(EXPORT['hvm'], 'hvm')
PROV = UNITS.uns['provenance']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['hvm'], PROV['fingerprints']['pass1']
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES71 = list(INP['genes'])
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
assert GENES71 == universe71[universe71.eligible.astype(bool)].gene.astype(str).tolist(), 'G1: universe differs from 71'
SETS71 = INP['gene_sets']
assert SETS71 == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G1: gene sets differ from 71'
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = np.asarray(INP['segment']).astype(str)
LIBRARY = np.asarray(INP['library'], float)
OBS = INP['obs']
IDENT = OBS['primary_identity'].astype(str).to_numpy()
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(POSITION)
GUARDS.append(f'G1: {len(GENES71):,} genes, {len(SETS71):,} sets and {len(POSITION):,} units equal 71')

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, SEGMENT, SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)
MOUSE_CUTS = sc.mean_transitions(TRANSITIONS, MOUSE)
UNITS_MOUSE = sc.to_segment_units(POSITION, MOUSE_CUTS)
GUARDS.append(f'G2: transitions reproduce 72\'s; mouse cuts {MOUSE_CUTS[0]:.3f}, {MOUSE_CUTS[1]:.3f}')

NATIVE, PANEL = {}, {}
for species, names in SPECIES.items():
    blocks = []
    for name in names:
        native = ad.read_h5ad(STAGE01 / f'{name}_units.h5ad')
        rows = np.flatnonzero(SPECIMEN == name)
        ids = OBS.polygon_id.astype(str).to_numpy()[rows]
        assert pd.Index(ids).isin(native.obs_names).all() and pd.Index(ids).is_unique, f'G3: {name} units missing'
        x = sparse.csr_matrix(native[list(ids)].X, dtype=float)
        export_x = sparse.csr_matrix(UNITS[OBS.index[rows]].X, dtype=float)
        symbols = UNITS.var_names if species == 'mouse' else pd.Index(UNITS.var.human_symbol.astype(str))
        cols = native.var_names.get_indexer(symbols)
        assert (cols >= 0).all() and (x[:, cols] != export_x).nnz == 0, f'G3: {name} counts differ from the export'
        PANEL[species] = pd.Index(native.var_names.astype(str))
        blocks.append((rows, x))
    NATIVE[species] = {'rows': np.concatenate([r for r, _ in blocks]),
                       'counts': sparse.vstack([x for _, x in blocks]).tocsr()}
GUARDS.append('G3: every export unit is in its stage-01 file; shared genes identical')

FROZEN_SHA['cellspectra_features.csv'] = sha256(FROZEN / 'cellspectra_features.csv')
for name, path in (('GO_0002931.tsv', QGO), ('supplementary_table_5.csv', CS / 'supplementary_table_5.csv'),
                   ('supplementary_table_8.csv', CS / 'supplementary_table_8.csv')):
    assert sha256(path) == FROZEN_SHA[name], f'G4: {name} changed'
EXTERNAL_SHA = {str(p.relative_to(project if str(p).startswith(str(project)) else data_root)): sha256(p) for p in
                [FROZEN / 'cellspectra_features.csv', QGO, CS / 'supplementary_table_5.csv', CS / 'supplementary_table_8.csv',
                 KNOWN / 'programs_hvm_members.csv', T81 / 'tables' / 'programs.csv',
                 *[LIBS / f'{lib}.json' for lib in ('GO_Biological_Process_2023', 'KEGG_2019_Mouse')]]}
GUARDS.append('G4: frozen feature list, QuickGO file and CellSpectra tables match their sha256')

INTEGRATION = set()
for name in ('pass1_labelled.h5ad', 'pass2.h5ad'):
    obj = ad.read_h5ad(RUN_HVM / '03_integration' / name, backed='r')
    INTEGRATION |= set(obj.var_names[obj.var['highly_variable_for_harmony'].to_numpy(bool)].astype(str))
    obj.file.close()
BUILT = INTEGRATION | set(LABEL_ORIENTATION)
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

## 3 · Binned profiles (healthy)

U_PT units; fixed 0.1 bins of mouse units on [0, 2.5); per specimen, each gene's log2 rate per bin minus its mean;
the reference is the median usable gene per bin. Human profiles use the human curve position (25 equal bins over
each section's 1–99% range), descriptive.

In [ ]:
U_PT = IDENT == 'Proximal_Tubule'


def species_rows(species, units='pt'):
    rows = NATIVE[species]['rows']
    keep = U_PT[rows] if units == 'pt' else np.ones(len(rows), bool)
    return rows[keep], NATIVE[species]['counts'][keep]


def detection_by(counts, specimen, names):
    return np.array([np.asarray((counts[specimen == n] > 0).mean(axis=0)).ravel() for n in names])


H = {}
for species, names in SPECIES.items():
    rows, x = species_rows(species)
    spec = SPECIMEN[rows]
    det = detection_by(x, spec, names)
    not_mt = ~PANEL[species].str.upper().str.startswith('MT-')
    expressed = (det >= DETECTION).all(axis=0) & not_mt
    genes = np.flatnonzero(expressed)
    H[species] = {'rows': rows, 'counts': x[:, genes], 'genes': list(PANEL[species][genes]), 'specimen': spec,
                  'library': LIBRARY[rows], 'segment': SEGMENT[rows], 'position': POSITION[rows],
                  'usable': (det[:, genes] >= USABLE).all(axis=0)}
    H[species]['index'] = pd.Series(np.arange(len(genes)), index=H[species]['genes'])

W86 = stored(T86, 'w_mouse_pt_shared')
W86_GENES = pd.read_csv(T86 / 'tables' / 'w_calls_mouse.csv', usecols=['gene']).gene.astype(str).tolist()
assert len(W86['alpha']) == len(H['mouse']['genes']) and W86_GENES == H['mouse']['genes'], \
    'G5: 86\'s mouse W run covers a different gene set'
GRID = stored(T71, f'gene_model_{DS}_species')['grid']
GRID_UNITS = sc.to_segment_units(GRID, MOUSE_CUTS)
GUARDS.append(f"G5: 86's cached W fits cover the {len(H['mouse']['genes']):,} PT-expressed mouse genes")

m = H['mouse']
m['units'] = UNITS_MOUSE[m['rows']]
BIN, CENTRES = pp.fixed_bins(m['units'])
NB = len(CENTRES)
m['bin'] = BIN
m['profiles'] = pp.bin_profiles(m['counts'], m['library'], BIN, NB, m['specimen'], MOUSE)
m['reference'] = {n: pp.reference_profile(m['profiles'][n], m['usable']) for n in MOUSE}
VALID = np.all([np.isfinite(m['reference'][n]) for n in MOUSE], axis=0)
assert VALID.all(), f'bins without {pp.MIN_UNITS} units in a mouse: {CENTRES[~VALID]}'
m['deviation'] = {n: m['profiles'][n] - m['reference'][n] for n in MOUSE}

hh = H['human']
hh['bin'] = np.full(len(hh['rows']), -1)
H_CENTRES = {}
for n in HUMAN:
    r = hh['specimen'] == n
    lo, hi = np.quantile(hh['position'][r], [.01, .99])
    e = np.linspace(lo, hi, 26)
    b = np.clip(np.searchsorted(e, hh['position'][r], side='right') - 1, 0, 24)
    b[(hh['position'][r] < lo) | (hh['position'][r] > hi)] = -1
    hh['bin'][r] = b
    H_CENTRES[n] = (e[:-1] + e[1:]) / 2
hh['profiles'] = {}
for n in HUMAN:
    r = hh['specimen'] == n
    hh['profiles'].update(pp.bin_profiles(hh['counts'][r], hh['library'][r], hh['bin'][r], 25, hh['specimen'][r], [n]))
hh['reference'] = {n: pp.reference_profile(hh['profiles'][n], hh['usable']) for n in HUMAN}
hh['deviation'] = {n: hh['profiles'][n] - hh['reference'][n] for n in HUMAN}
HUMAN_CUTS = {n: TRANSITIONS[TRANSITIONS.specimen.eq(n)].sort_values('transition').position.to_numpy() for n in HUMAN}
print({s: (len(H[s]['genes']), int(H[s]['usable'].sum())) for s in SPECIES}, 'PT-expressed, usable genes')

# Cluster view: label pseudobulk on U_PT, 81's segment_profile per mouse
LABEL_RATES = {}
rates = segment_pseudobulk(m['counts'], m['library'], m['specimen'], m['segment'], {s: [f'PT-{s}'] for s in SEGMENTS},
                           genes=m['genes'])
for n in MOUSE:
    LABEL_RATES[n] = pd.DataFrame({s: rates[s][n] for s in SEGMENTS})
LABEL_MEDIAN = {}
for n in MOUSE:
    c = LABEL_RATES[n].sub(LABEL_RATES[n].mean(axis=1), axis=0)
    LABEL_MEDIAN[n] = c.median()


def cluster_view(members):
    """Per mouse, 81's segment_profile of the members (log2); and the call."""
    vals = np.array([pf.segment_profile(LABEL_RATES[n], members).to_numpy() for n in MOUSE])
    return vals, pp.segment_call(vals)


def members_of(genes, species='mouse', usable=True):
    idx = H[species]['index'].reindex([g for g in dict.fromkeys(genes)]).dropna().astype(int).to_numpy()
    return idx[H[species]['usable'][idx]] if usable else idx


def mouse_profiles(member_rows):
    """Program profile per mouse (programs x bins) from lists of usable-gene row indices."""
    return {n: pp.program_profiles(m['deviation'][n], member_rows, min_members=1) for n in MOUSE}


def human_symbols(mouse_genes):
    mp = orthologs.drop_duplicates('mouse_symbol').set_index('mouse_symbol').human_symbol.astype(str)
    return [mp[g] for g in mouse_genes if g in mp.index]

## 4 · T · Can the curve show it? (semi-synthetic)

15 usable mouse genes per program, NB counts with each gene's own dispersion and the specimen's typical shape, a
known effect f(u) written at a generation position: the observed position (ideal) or a fold refit's position
(realistic: the coordinate's own error). Analysed exactly as the data.

In [ ]:
def refit_units():
    """Mouse units of every hvm unit in each fold refit (its own support scaling and mouse transitions)."""
    out = {}
    ids = pd.Index(UNITS.obs_names.astype(str))
    obs = UNITS.obs
    seg_all, spec_all = obs.segment_class.astype(str).to_numpy(), obs['sample'].astype(str).to_numpy()
    rows_in_export = ids.get_indexer(OBS.index.astype(str))
    for k in range(3):
        p = stored(T72, f'refit_hvm_fold{k}')
        z = refit_module.refit_position(p, ids)
        if not refit_module.runs_s1_to_s3(z, spec_all, seg_all):
            continue
        finite = np.isfinite(z)
        mask, kept = support_positions(z[finite], spec_all[finite], seg_all[finite])
        lo, hi = z[finite][mask].min(), z[finite][mask].max()
        pos = np.full(len(z), np.nan)
        pos[finite] = (z[finite] - lo) / (hi - lo)
        sub = pos[rows_in_export]
        ok = np.isfinite(sub)
        tr = logistic_transitions(sub[ok], SEGMENT[ok], SPECIMEN[ok])
        cuts = sc.mean_transitions(tr, MOUSE)
        out[k] = sc.to_segment_units(sub, cuts)
    return out


REFIT = refit_units()
FOLDS = sorted(REFIT)
cover = {k: float(np.isfinite(v[m['rows']]).mean()) for k, v in REFIT.items()}
assert len(FOLDS) >= 2 and min(cover.values()) >= .9, f'G6: fold refit coverage {cover}'
GUARDS.append(f'G6: fold refits {FOLDS} cover {min(cover.values()):.3f}–{max(cover.values()):.3f} of mouse U_PT units')
if len(FOLDS) < 3:
    READING.append(f'T realistic arm: fold refits {FOLDS} only; the others fail 72\'s S1→S3 orientation guard (72 also '
                   'used folds 0 and 1), so replicate r uses fold FOLDS[r mod 2] instead of r mod 3.')

SHAPES = {f'bump c={c} w={w}': ('bump', c, w) for c in (1., 2.) for w in (.25, .5, 1.)}
SHAPES.update({f'step-matched c={c} w={w}': ('step', c, w) for c in (1., 2.) for w in (.25, .5, 1.)})
SHAPES.update({f'ramp c=1.0 w={w}': ('ramp', 1., w) for w in (.1, .5, 1.5)})
SHAPES.update({f'ramp c={c} w=0.5': ('ramp', c, .5) for c in (.6, 1.4)})
SHAPES.update({f'local c={c}': ('local', c, .4) for c in (.3, .7, 1.3, 1.7)})
SHAPES['flat'] = ('flat', np.nan, np.nan)

RATE = {n: np.asarray(m['counts'][m['specimen'] == n].sum(axis=0)).ravel() / m['library'][m['specimen'] == n].sum()
        for n in MOUSE}
TAU = {}
for n in MOUSE:
    shape = W86[f'shape|{n}']
    inside = np.isfinite(shape)
    TAU[n] = (GRID_UNITS[inside], shape[inside] - shape[inside].mean())
USABLE_ROWS = np.flatnonzero(m['usable'])
ALPHA = W86['alpha']


def effect_values(kind, c, w, u, label, spec):
    if kind == 'bump' or kind == 'local':
        return pp.bump(u, c, w)
    if kind == 'step':
        return pp.matched_step(pp.bump(u, c, w), label, spec)
    if kind == 'ramp':
        return pp.ramp(u, c, w)
    return np.zeros(len(u))


def simulate(kind, c, w, amplitude, arm, rep):
    rng = np.random.default_rng([SEED, rep, int(amplitude * 10), zlib.crc32(f'{kind}|{c}|{w}|{arm}'.encode())])
    genes = rng.choice(USABLE_ROWS, SIM_GENES, replace=False)
    a = amplitude * rng.uniform(.5, 1.5, SIM_GENES)
    u_gen = m['units'] if arm == 'ideal' else REFIT[FOLDS[rep % len(FOLDS)]][m['rows']]
    keep = np.isfinite(u_gen)
    prof, label_vals = {}, {}
    for n in MOUSE:
        r = (m['specimen'] == n) & keep
        u = u_gen[r]
        f = effect_values(kind, c, w, u, m['segment'][r], m['specimen'][r])
        tau = np.log(2) * np.interp(u, *TAU[n])
        x = pp.simulate_counts(m['library'][r], RATE[n][genes], tau, np.log(2) * a[:, None] * f[None, :], ALPHA[genes], rng)
        inb = m['bin'][r] >= 0
        y = ts.binned_log_rates(sparse.csr_matrix(x[inb]), m['library'][r][inb], m['bin'][r][inb], NB)
        y = y - y.mean(axis=1, keepdims=True)
        prof[n] = np.median(y - m['reference'][n], axis=0)
        lab = pd.DataFrame(x, columns=range(SIM_GENES)).groupby(m['segment'][r]).sum()
        lib = pd.Series(m['library'][r]).groupby(m['segment'][r]).sum()
        lr = np.log2(lab.add(.5).div(lib, axis=0)).T.rename(columns=lambda s: s.replace('PT-', ''))[list(SEGMENTS)]
        cv = lr.sub(lr.mean(axis=1), axis=0) - LABEL_MEDIAN[n]
        label_vals[n] = cv.median().to_numpy()
    return prof, label_vals


def run_simulation():
    rows, profiles = [], []
    for name, (kind, c, w) in SHAPES.items():
        for amplitude in AMPLITUDES:
            for arm in ('ideal', 'realistic'):
                for rep in range(N_SIM):
                    prof, lab = simulate(kind, c, w, amplitude, arm, rep)
                    rows.append((name, kind, c, w, amplitude, arm, rep))
                    profiles.append(np.concatenate([prof[n] for n in MOUSE] + [lab[n] for n in MOUSE]))
    return {'meta': np.array([json.dumps(r) for r in rows]), 'values': np.array(profiles)}


code_key = digest([NOTEBOOK_LOGIC_VERSION, Path(pp.__file__).read_text()])
SECONDS = {}


def cached(stage, compute, *, params=None, inputs=None):
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


SIM = cached('simulation', run_simulation,
             params={'shapes': json.dumps(SHAPES), 'amplitudes': AMPLITUDES, 'n': N_SIM, 'genes': SIM_GENES, 'seed': SEED},
             inputs={'counts': digest(m['counts']), 'units': m['units'], 'refit': np.vstack([REFIT[k][m['rows']] for k in FOLDS]),
                     'alpha': ALPHA})
meta = pd.DataFrame([json.loads(s) for s in SIM['meta']], columns=['shape', 'kind', 'c', 'w', 'A', 'arm', 'rep'])
V = SIM['values']
SIM_PROF = {n: V[:, i * NB:(i + 1) * NB] for i, n in enumerate(MOUSE)}
SIM_LAB = {n: V[:, 2 * NB + 3 * i:2 * NB + 3 * (i + 1)] for i, n in enumerate(MOUSE)}
print(f'T: {len(meta):,} synthetic programs ({SECONDS["simulation"]:.0f} s)')


def boundary_calls(profiles_by_mouse, centres, *, half=pp.HALF, flank=pp.FLANK):
    """Per program: E and T at both boundaries per mouse, and the agreed boundary-program call."""
    out = {}
    for k in (1, 2):
        E = {n: pp.boundary_excess(profiles_by_mouse[n], centres, k, half=half, flank=flank) for n in profiles_by_mouse}
        names = list(profiles_by_mouse)
        out[f'E{k}'] = np.minimum(*[E[n][0] for n in names])
        out[f'T{k}'] = np.minimum(*[E[n][1] for n in names])
        for n in names:
            out[f'E{k} {n}'], out[f'T{k} {n}'] = E[n]
    frame = pd.DataFrame(out)
    frame['boundary'] = (frame[['E1', 'T1', 'E2', 'T2']] >= EFFECT).any(axis=1)
    best = frame[['E1', 'T1', 'E2', 'T2']].idxmax(axis=1)
    frame['best'] = best.where(frame.boundary, '')
    frame['agreed excess'] = frame[['E1', 'T1', 'E2', 'T2']].max(axis=1)
    return frame


sim_calls = boundary_calls(SIM_PROF, CENTRES)
meta = meta.join(sim_calls[['E1', 'T1', 'E2', 'T2', 'boundary', 'best']])
meta['cluster call'] = [pp.segment_call(np.vstack([SIM_LAB[n][i] for n in MOUSE])) for i in range(len(meta))]
mean_prof = (SIM_PROF[MOUSE[0]] + SIM_PROF[MOUSE[1]]) / 2

# boundary sensitivity / false positives: the call at the bump's own boundary
own = np.where(meta.c.eq(1.), meta.E1, np.where(meta.c.eq(2.), meta.E2, np.nan))
meta['own boundary'] = own >= EFFECT
# ramps: edges on the mean profile
ramp_rows = np.flatnonzero(meta.kind.eq('ramp'))
_, ramp_edges = ts.fit_edges(CENTRES, mean_prof[ramp_rows])
main = ramp_edges.loc[ramp_edges.groupby('profile').amplitude.apply(lambda a: a.abs().idxmax())] if len(ramp_edges) else ramp_edges
main = main.set_index('profile')
meta['edge position'], meta['edge width'] = np.nan, np.nan
meta.loc[meta.index[ramp_rows[main.index]], 'edge position'] = main.position.to_numpy()
meta.loc[meta.index[ramp_rows[main.index]], 'edge width'] = main.width.to_numpy()
meta['edge class'] = meta['edge width'].map(ts.edge_class)
# local: peak position
lr_rows = meta.kind.eq('local')
meta['peak position'] = np.where(lr_rows, CENTRES[np.argmax(mean_prof, axis=1)], np.nan)
meta['correct half'] = np.where(lr_rows, np.floor(meta['peak position'] * 2) == np.floor(meta.c * 2), np.nan)

T_SUMMARY = (meta.groupby(['shape', 'kind', 'c', 'w', 'A', 'arm'], dropna=False)
             .agg(n=('rep', 'size'), boundary=('boundary', 'mean'), own_boundary=('own boundary', 'mean'),
                  cluster_none=('cluster call', lambda s: (s == 'none').mean()),
                  cluster_one=('cluster call', lambda s: s.isin(SEGMENTS).mean()),
                  cluster_two=('cluster call', lambda s: s.str.contains(r'\+').mean()),
                  edge_width=('edge width', 'median'), sharp=('edge class', lambda s: (s == 'sharp').mean()),
                  gradual=('edge class', lambda s: (s == 'gradual').mean()),
                  edge_error=('edge position', lambda s: np.nan),
                  peak_error=('peak position', 'median'), correct_half=('correct half', 'mean'))
             .reset_index())
ramp_err = meta[meta.kind.eq('ramp')].assign(err=lambda d: (d['edge position'] - d.c).abs()).groupby(['shape', 'A', 'arm']).err.median()
T_SUMMARY['edge_error'] = [ramp_err.get((r.shape, r.A, r.arm), np.nan) for r in T_SUMMARY.itertuples()]
loc_err = meta[lr_rows].assign(err=lambda d: (d['peak position'] - d.c).abs()).groupby(['shape', 'A', 'arm']).err.median()
T_SUMMARY['peak_error'] = [loc_err.get((r.shape, r.A, r.arm), np.nan) for r in T_SUMMARY.itertuples()]
T_SUMMARY.to_csv(tables / 't_operating_characteristics.csv', index=False)


def at(shape, A=1., arm='realistic', col='own_boundary'):
    return float(T_SUMMARY[(T_SUMMARY['shape'] == shape) & (T_SUMMARY.A == A) & (T_SUMMARY.arm == arm)][col].iloc[0])


SHOWABLE = {
    'A1 S1/S2': at('bump c=1.0 w=0.5') >= .8 and max(at('step-matched c=1.0 w=0.5', col='boundary'), at('flat', col='boundary')) <= .05,
    'A1 S2/S3': at('bump c=2.0 w=0.5') >= .8 and max(at('step-matched c=2.0 w=0.5', col='boundary'), at('flat', col='boundary')) <= .05,
    'A2': at('ramp c=1.0 w=0.1', col='sharp') >= .8 and at('ramp c=1.0 w=1.5', col='gradual') >= .8,
    'A3': np.mean([at(f'local c={c}', col='correct_half') for c in (.3, .7, 1.3, 1.7)]) >= .8,
}
A3_BY_SEGMENT = {s: np.mean([at(f'local c={c}', col='correct_half') for c in cs]) for s, cs in (('S1', (.3, .7)), ('S2', (1.3, 1.7)))}
print('Showable (realistic arm, A = 1):', SHOWABLE, '· A3 by segment', {k: round(v, 2) for k, v in A3_BY_SEGMENT.items()})
display(T_SUMMARY[T_SUMMARY.A.eq(1.)].drop(columns=['kind', 'c', 'w', 'A', 'n']).round(3))

## 5 · H1 · Boundary programs (primary)

71's sets with ≥ 10 usable members. A set is a boundary program when its window at a boundary rises above (or
falls below) both flanks by ≥ 0.25 log2 in both mice. Null: gene labels permuted within expression deciles.

In [ ]:
LOG_CP = np.log2(np.mean([RATE[n] * 1e4 for n in MOUSE], axis=0) + 1e-6)
U_IDX = np.flatnonzero(m['usable'])
pos_in_usable = pd.Series(np.arange(len(U_IDX)), index=U_IDX)
STRATA = pd.qcut(LOG_CP[U_IDX], 10, labels=False, duplicates='drop')
SET_IDS = [s for s in SETS71]
SET_MEMBERS = {s: members_of(SETS71[s]) for s in SET_IDS}
SET_IDS = [s for s in SET_IDS if len(SET_MEMBERS[s]) >= MIN_SET]
SET_U = [pos_in_usable[SET_MEMBERS[s]].to_numpy() for s in SET_IDS]
print(f'{len(SET_IDS):,} of {len(SETS71):,} sets have ≥ {MIN_SET} usable members')


def scan(deviation, member_u, centres, perms, *, half=pp.HALF, flank=pp.FLANK, amplitude=True):
    """Observed boundary calls and, per permutation, the number of boundary programs; per set the 95th percentile of
    its permuted amplitude in each mouse."""
    Du = {n: deviation[n][U_IDX] for n in MOUSE}
    obs = {n: np.vstack([np.median(Du[n][u], axis=0) for u in member_u]) for n in MOUSE}
    calls = boundary_calls(obs, centres, half=half, flank=flank)
    null_hits = np.zeros((len(perms), len(member_u)), bool)
    amp95 = np.full((len(member_u), len(MOUSE)), np.nan)
    for i, u in enumerate(member_u):
        idx = perms[:, u]
        pm = {n: np.median(Du[n][idx], axis=1) for n in MOUSE}
        c = boundary_calls(pm, centres, half=half, flank=flank)
        null_hits[:, i] = c.boundary.to_numpy()
        if amplitude:
            amp95[i] = [np.quantile(np.ptp(pm[n], axis=1), .95) for n in MOUSE]
    return {'obs': obs, 'calls': calls, 'null_count': null_hits.sum(axis=1), 'amp95': amp95}


PERMS = pp.strata_permutations(STRATA, N_PERM, np.random.default_rng(SEED))


def h1_compute():
    r = scan(m['deviation'], SET_U, CENTRES, PERMS)
    return {'obs_A2': r['obs'][MOUSE[0]], 'obs_A4': r['obs'][MOUSE[1]], 'null_count': r['null_count'], 'amp95': r['amp95']}


H1 = cached('h1_scan', h1_compute, params={'n_perm': N_PERM, 'seed': SEED, 'half': pp.HALF, 'flank': pp.FLANK},
            inputs={'counts': digest(m['counts']), 'units': m['units'], 'sets': json.dumps(SET_IDS)})
OBS_PROF = {MOUSE[0]: H1['obs_A2'], MOUSE[1]: H1['obs_A4']}
CALLS = boundary_calls(OBS_PROF, CENTRES).set_index(pd.Index(SET_IDS, name='set'))
C_OBS = int(CALLS.boundary.sum())
P_H1 = (1 + int((H1['null_count'] >= C_OBS).sum())) / (N_PERM + 1)
BOUNDARY_SETS = CALLS.index[CALLS.boundary].tolist()
GROUPS_B = pp.redundancy_groups({s: [m['genes'][i] for i in SET_MEMBERS[s]] for s in BOUNDARY_SETS}) if BOUNDARY_SETS else pd.Series(dtype=int)
N_GROUPS_B = int(GROUPS_B.nunique()) if len(GROUPS_B) else 0
H1_HOLDS = P_H1 <= .05 and N_GROUPS_B >= 3
print(f'H1: {C_OBS} boundary programs ({N_GROUPS_B} redundancy groups); null mean {H1["null_count"].mean():.2f}, '
      f'95th {np.quantile(H1["null_count"], .95):.0f}; p = {P_H1:.4f}; holds: {H1_HOLDS}')

CHEN = pd.read_csv(T84 / 'tables' / 'segment_pathways.csv')
CHEN_CALL = CHEN[CHEN.fdr <= .05].groupby('pathway').segment.apply(lambda s: '+'.join(sorted(set(s))))


def short(set_id):
    lib, name = set_id.split('::', 1)
    return {'Reactome_2022': 'R', 'KEGG_2019_Mouse': 'K', 'MSigDB_Hallmark_2020': 'H',
            'GO_Biological_Process_2023': 'GO'}.get(lib, lib) + ': ' + name.split(' R-HSA')[0]


def gene_boundary(idx_rows, kind, k):
    """Members that are boundary genes themselves (agreed) for the set's kind and boundary."""
    prof = {n: m['deviation'][n][idx_rows] for n in MOUSE}
    e = {n: pp.boundary_excess(prof[n], CENTRES, k) for n in MOUSE}
    j = 0 if kind == 'E' else 1
    hit = (e[MOUSE[0]][j] >= EFFECT) & (e[MOUSE[1]][j] >= EFFECT)
    return [m['genes'][i] for i in np.asarray(idx_rows)[hit]]


rows = []
for s in BOUNDARY_SETS:
    c = CALLS.loc[s]
    kind, k = c.best[0], int(c.best[1])
    idx = SET_MEMBERS[s]
    vals, call = cluster_view([m['genes'][i] for i in idx])
    protected = np.array([i for i in idx if m['genes'][i] not in BUILT])
    if len(protected) >= MIN_SET:
        pprof = {n: np.median(m['deviation'][n][protected], axis=0)[None] for n in MOUSE}
        pc = boundary_calls(pprof, CENTRES).iloc[0]
        protected_excess = float(pc[f'{kind}{k}'])
    else:
        protected_excess = np.nan
    core = gene_boundary(idx, kind, k)
    rows.append({'set': s, 'name': short(s), 'boundary': f'S{k}/S{k + 1}', 'kind': 'peak' if kind == 'E' else 'trough',
                 'excess (agreed)': c[f'{kind}{k}'], f'excess {MOUSE[0]}': c[f'{kind}{k} {MOUSE[0]}'],
                 f'excess {MOUSE[1]}': c[f'{kind}{k} {MOUSE[1]}'], 'members': len(idx),
                 'cluster call': call, **{f'label {seg}': float(vals.mean(axis=0)[j]) for j, seg in enumerate(SEGMENTS)},
                 'Chen call': CHEN_CALL.get(s, 'none'), 'excess without built genes': protected_excess,
                 'core genes': ', '.join(core[:12]), 'n core': len(core), 'group': int(GROUPS_B.get(s, -1))})
BOUNDARY_TABLE = pd.DataFrame(rows).sort_values(['boundary', 'excess (agreed)'], ascending=[True, False]) if rows else pd.DataFrame()
BOUNDARY_TABLE.to_csv(tables / 'h1_boundary_programs.csv', index=False)
CALLS.to_csv(tables / 'h1_set_calls.csv')
pd.DataFrame({'null count': H1['null_count']}).to_csv(tables / 'h1_null_counts.csv', index=False)
display(BOUNDARY_TABLE)

### Sensitivities (descriptive; 200 permutations each)

In [ ]:
def sens_compute():
    out = {}
    perms = PERMS[:N_PERM_SENS]
    for half in (.15, .35):
        r = scan(m['deviation'], SET_U, CENTRES, perms, half=half, amplitude=False)
        out[f'half {half}'] = np.array([r['calls'].boundary.sum(), *r['null_count']])
    # bin width 0.2
    b2, c2 = pp.fixed_bins(m['units'], width=.2)
    prof2 = pp.bin_profiles(m['counts'], m['library'], b2, len(c2), m['specimen'], MOUSE)
    dev2 = {n: prof2[n] - pp.reference_profile(prof2[n], m['usable']) for n in MOUSE}
    r = scan(dev2, SET_U, c2, perms, amplitude=False)
    out['bin 0.2'] = np.array([r['calls'].boundary.sum(), *r['null_count']])
    # built genes removed from members
    keep = [np.array([j for j in u if m['genes'][U_IDX[j]] not in BUILT]) for u in SET_U]
    ok = [i for i, u in enumerate(keep) if len(u) >= MIN_SET]
    r = scan(m['deviation'], [keep[i] for i in ok], CENTRES, perms, amplitude=False)
    out['built genes removed'] = np.array([r['calls'].boundary.sum(), *r['null_count']])
    # all PT-labelled units
    rows_all, x_all = species_rows('mouse', units='all')
    xa = x_all[:, [PANEL['mouse'].get_loc(g) for g in m['genes']]]
    ua = UNITS_MOUSE[rows_all]
    ba, _ = pp.fixed_bins(ua)
    profa = pp.bin_profiles(xa, LIBRARY[rows_all], ba, NB, SPECIMEN[rows_all], MOUSE)
    deva = {n: profa[n] - pp.reference_profile(profa[n], m['usable']) for n in MOUSE}
    r = scan(deva, SET_U, CENTRES, perms, amplitude=False)
    out['U_all'] = np.array([r['calls'].boundary.sum(), *r['null_count']])
    return out


SENS = cached('h1_sensitivities', sens_compute, params={'n_perm': N_PERM_SENS, 'seed': SEED},
              inputs={'counts': digest(m['counts']), 'units': m['units'], 'sets': json.dumps(SET_IDS)})
SENS_TABLE = pd.DataFrame([{'sensitivity': k, 'boundary programs': int(v[0]), 'null mean': float(v[1:].mean()),
                            'p': (1 + int((v[1:] >= v[0]).sum())) / (len(v))} for k, v in SENS.items()])
SENS_TABLE.to_csv(tables / 'h1_sensitivities.csv', index=False)
display(SENS_TABLE)

## 6 · E1 · Positional programs

Eligible: amplitude above the set's own 95th permuted percentile in each mouse, and the two mice's profiles at
r ≥ 0.5. Redundancy groups (Jaccard ≥ 0.5) give representatives; average linkage on 1 − r; K by silhouette.

In [ ]:
AMP = {n: np.ptp(OBS_PROF[n], axis=1) for n in MOUSE}
R_MICE = np.array([np.corrcoef(OBS_PROF[MOUSE[0]][i], OBS_PROF[MOUSE[1]][i])[0, 1] for i in range(len(SET_IDS))])
ELIG = (AMP[MOUSE[0]] > H1['amp95'][:, 0]) & (AMP[MOUSE[1]] > H1['amp95'][:, 1]) & (R_MICE >= R_ELIGIBLE)
ELIG_IDS = [s for s, e in zip(SET_IDS, ELIG) if e]
groups = pp.redundancy_groups({s: [m['genes'][i] for i in SET_MEMBERS[s]] for s in ELIG_IDS})
rep_of = {}
for g, ss in groups.groupby(groups).groups.items():
    rep_of[g] = sorted(ss, key=lambda s: (-len(SET_MEMBERS[s]), s))[0]
REPS = [rep_of[g] for g in sorted(rep_of)]
ri = [SET_IDS.index(s) for s in REPS]
MEAN_PROF = (OBS_PROF[MOUSE[0]] + OBS_PROF[MOUSE[1]]) / 2
labels, K, SIL = pp.cluster_profiles(MEAN_PROF[ri])
lab_a, _, _ = pp.cluster_profiles(OBS_PROF[MOUSE[0]][ri], k=K)
lab_b, _, _ = pp.cluster_profiles(OBS_PROF[MOUSE[1]][ri], k=K)
ARI = adjusted_rand_score(lab_a, lab_b)
lab6_a, _, _ = pp.cluster_profiles(OBS_PROF[MOUSE[0]][ri], k=6)
lab6_b, _, _ = pp.cluster_profiles(OBS_PROF[MOUSE[1]][ri], k=6)
ARI6 = adjusted_rand_score(lab6_a, lab6_b)
sizes = pd.Series(labels).value_counts()
programs = [k for k in sizes.index if sizes[k] >= 3]
centroids = {k: MEAN_PROF[ri][labels == k].mean(axis=0) for k in programs}
order = sorted(programs, key=lambda k: CENTRES[np.argmax(centroids[k])])
PROGRAM_NAME = {k: f'P{i + 1}' for i, k in enumerate(order)}
REP_TABLE = pd.DataFrame({'set': REPS, 'name': [short(s) for s in REPS], 'cluster': labels,
                          'program': [PROGRAM_NAME.get(k, '') for k in labels],
                          'group size': [int((groups == groups[s]).sum()) for s in REPS],
                          'members': [len(SET_MEMBERS[s]) for s in REPS],
                          'r mice': R_MICE[ri], 'Chen call': [CHEN_CALL.get(s, 'none') for s in REPS],
                          'cluster call': [cluster_view([m['genes'][i] for i in SET_MEMBERS[s]])[1] for s in REPS]})
print(f'E1: {int(ELIG.sum())} eligible sets in {len(REPS)} redundancy groups; K = {K} (silhouette {SIL[K]:.3f}); '
      f'{len(programs)} programs with ≥ 3 representatives; ARI between mice {ARI:.3f} (K = 6: {ARI6:.3f}); '
      f'replicate: {ARI >= ARI_MIN}')


def describe(profile):
    d = pf.shape_descriptors(profile, CENTRES)
    if d['shape'] in ('mixed',) or (d['shape'] == 'flat'):
        j = int(np.argmax(np.abs(profile - np.median(profile))))
        d['position'] = float(CENTRES[j])
    return d


PROGRAM_TABLE = []
for k in order:
    sel = REP_TABLE[REP_TABLE.cluster.eq(k)]
    d = describe(centroids[k])
    PROGRAM_TABLE.append({'program': PROGRAM_NAME[k], 'representatives': len(sel), 'sets': int(sel['group size'].sum()),
                          'max at': float(CENTRES[np.argmax(centroids[k])]), 'min at': float(CENTRES[np.argmin(centroids[k])]),
                          'shape': d['shape'], 'position': d['position'], 'amplitude': float(np.ptp(centroids[k])),
                          'cluster calls': '; '.join(f'{c} {n}' for c, n in sel['cluster call'].value_counts().items()),
                          'Chen calls': '; '.join(f'{c} {n}' for c, n in sel['Chen call'].value_counts().items()),
                          'examples': '; '.join(sel.sort_values('members', ascending=False).name.head(6))})
PROGRAM_TABLE = pd.DataFrame(PROGRAM_TABLE)
REP_TABLE.to_csv(tables / 'e1_representatives.csv', index=False)
PROGRAM_TABLE.to_csv(tables / 'e1_programs.csv', index=False)
display(PROGRAM_TABLE)

# Sensitivity: K fixed at 6 (protocol)
lab6, _, _ = pp.cluster_profiles(MEAN_PROF[ri], k=6)
REP_TABLE['cluster K6'] = lab6
K6_ORDER = sorted(set(lab6), key=lambda k: CENTRES[np.argmax(MEAN_PROF[ri][lab6 == k].mean(axis=0))])
K6_TABLE = []
for j, k in enumerate(K6_ORDER):
    sel = REP_TABLE[REP_TABLE['cluster K6'].eq(k)]
    cen = MEAN_PROF[ri][lab6 == k].mean(axis=0)
    d = describe(cen)
    K6_TABLE.append({'cluster': f'K6-{j + 1}', 'representatives': len(sel), 'sets': int(sel['group size'].sum()),
                     'max at': float(CENTRES[np.argmax(cen)]), 'min at': float(CENTRES[np.argmin(cen)]),
                     'shape': d['shape'], 'position': d['position'], 'amplitude': float(np.ptp(cen)),
                     'K3 program': '; '.join(f'{c or "unassigned"} {n}' for c, n in sel.program.value_counts().items()),
                     'Chen calls': '; '.join(f'{c} {n}' for c, n in sel['Chen call'].value_counts().items()),
                     'examples': '; '.join(sel.sort_values('members', ascending=False).name.head(6))})
K6_TABLE = pd.DataFrame(K6_TABLE)
K6_TABLE.to_csv(tables / 'e1_sensitivity_k6.csv', index=False)
REP_TABLE.to_csv(tables / 'e1_representatives.csv', index=False)
display(K6_TABLE)

## 7 · E2 · What the cluster view says about the same sets

In [ ]:
E2_ROWS = []
for s in ELIG_IDS:
    i = SET_IDS.index(s)
    prof = MEAN_PROF[i]
    call = cluster_view([m['genes'][j] for j in SET_MEMBERS[s]])[1]
    d = describe(prof)
    pos = d['position']
    if call == 'none':
        cat = 'missed'
    elif CALLS.loc[s, 'boundary'] or min(abs(pos - 1), abs(pos - 2)) <= .25:
        cat = 'boundary'
    else:
        seg = SEGMENTS[int(np.clip(np.floor(pos), 0, 2))]
        cat = 'agree' if seg in call.split('+') else 'mislocated'
    E2_ROWS.append({'set': s, 'name': short(s), 'shape': d['shape'], 'position': pos, 'width': d['width'],
                    'cluster call': call, 'Chen call': CHEN_CALL.get(s, 'none'), 'category': cat})
E2 = pd.DataFrame(E2_ROWS)
E2.to_csv(tables / 'e2_cluster_vs_curve.csv', index=False)
E2_COUNTS = E2.category.value_counts().reindex(['agree', 'boundary', 'mislocated', 'missed'], fill_value=0)
print('E2:', E2_COUNTS.to_dict(), f'· not "agree": {1 - E2_COUNTS["agree"] / max(len(E2), 1):.2f}')

## 8 · E3 · CellSpectra's programs along the healthy PT

In [ ]:
def library_members(key, universe):
    """Mouse members of one frozen feature on ``universe`` (GO BP 2023 / KEGG 2019 via the ortholog map; QuickGO)."""
    if key.startswith('QuickGO::'):
        q = pd.read_csv(QGO, sep='\t')
        return sorted(set(q.SYMBOL.astype(str)) & set(universe))
    lib, name = key.split('::', 1)
    sets = json.loads((LIBS / f'{lib}.json').read_text())
    cov = build_pathway_membership({name: sets[name]}, list(universe), ortholog_map=orthologs, library_name=lib,
                                   tested=list(universe), min_genes=1)
    return list(cov.genes_present.iloc[0]) if len(cov) else []


FEATURES = pd.read_csv(FROZEN / 'cellspectra_features.csv')
prog81 = pd.read_csv(T81 / 'tables' / 'programs.csv')
known = pd.read_csv(KNOWN / 'programs_hvm_members.csv')
E3_PROGRAMS = {}
for pid in ['S-H1', 'S-H2', 'S-H3', 'S-M1', 'S-M2', 'S-M3']:
    E3_PROGRAMS[pid] = ('species list', known[known.program_id.eq(pid)].gene.astype(str).tolist())
for r in prog81[prog81.program_id.str.match(r'^(U-|F-)')].itertuples():
    E3_PROGRAMS[r.program_id] = ('segment-only list' if r.program_id.startswith('U-') and r.program_id[-1] != 'C'
                                 else 'core list' if r.program_id.startswith('U-') else 'conserved function',
                                 [g.strip() for g in str(r.genes).split(',')])
for r in FEATURES.itertuples():
    E3_PROGRAMS[r.id] = (f'disease: {r.family}', library_members(r.library_key, m['genes']))
E3_ROWS, E3_PROF = [], {}
for pid, (kind, genes) in E3_PROGRAMS.items():
    idx = members_of(genes)
    if len(idx) < MIN_FEATURE:
        E3_ROWS.append({'program': pid, 'kind': kind, 'members': len(idx), 'shape': 'too few members'})
        continue
    u = pos_in_usable[idx].to_numpy()
    prof = {n: np.median(m['deviation'][n][idx], axis=0) for n in MOUSE}
    c = boundary_calls({n: prof[n][None] for n in MOUSE}, CENTRES).iloc[0]
    pm = {n: np.median(m['deviation'][n][U_IDX][PERMS[:N_PERM_SENS][:, u]], axis=1) for n in MOUSE}
    null_c = boundary_calls(pm, CENTRES)['agreed excess'].to_numpy()
    E3_PROF[pid] = prof
    mean = (prof[MOUSE[0]] + prof[MOUSE[1]]) / 2
    d = describe(mean)
    vals, call = cluster_view([m['genes'][i] for i in idx])
    near = {PROGRAM_NAME[k]: np.corrcoef(mean, centroids[k])[0, 1] for k in order}
    best = max(near, key=near.get) if near else ''
    hm = members_of(human_symbols([m['genes'][i] for i in idx]), 'human')
    hr = np.nan
    if len(hm) >= MIN_FEATURE:
        hp = [np.median(hh['deviation'][n][hm], axis=0) for n in HUMAN]
        ok = np.isfinite(hp[0]) & np.isfinite(hp[1])
        hr = float(np.corrcoef(hp[0][ok], hp[1][ok])[0, 1])
    E3_ROWS.append({'program': pid, 'kind': kind, 'members': len(idx), 'shape': d['shape'], 'position': d['position'],
                    'width': d['width'], 'amplitude': float(np.ptp(mean)),
                    'r mice': float(np.corrcoef(prof[MOUSE[0]], prof[MOUSE[1]])[0, 1]),
                    'boundary': c.best if c.boundary else '', 'agreed excess': float(c['agreed excess']),
                    'boundary p': (1 + int((null_c >= c['agreed excess']).sum())) / (len(null_c) + 1),
                    'cluster call': call, **{f'label {seg}': float(vals.mean(axis=0)[j]) for j, seg in enumerate(SEGMENTS)},
                    'nearest program': best if near and near[best] >= R_NEAREST else '',
                    'r nearest': near.get(best, np.nan), 'human members': len(hm), 'r human sections': hr})
E3 = pd.DataFrame(E3_ROWS)
E3.to_csv(tables / 'e3_cellspectra_programs.csv', index=False)
display(E3.round(3))

## 9 · E4 · CellSpectra's PT disease features in the IR mice

AKI export, U_PT units of the four mice, the full panel (detection ≥ 2% in each mouse). Δ = IR − control per bin,
centred by the median gene; per pair and per relabeling. Coordination: CellSpectra's R² between specimens, by
position, at matched depth.

In [ ]:
AKI = read_pt_units(EXPORT['aki'], 'aki')
assert AKI.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['aki']
aobs = AKI.obs
a_spec_all = aobs['sample'].astype(str).to_numpy()
a_seg_all = aobs.segment_class.astype(str).to_numpy()
a_mask, a_pos = support_positions(aobs.pseudospace.to_numpy(float), a_spec_all, a_seg_all)
a_spec, a_seg = a_spec_all[a_mask], a_seg_all[a_mask]
a_ident = aobs.primary_identity.astype(str).to_numpy()[a_mask]
a_tr = logistic_transitions(a_pos, a_seg, a_spec)
for n in CONTROLS:
    t = a_tr[a_tr.specimen.eq(n)].sort_values('transition').position.to_numpy()
    assert len(t) == 2 and t[0] < t[1], f'G7: {n} transitions out of order'
AKI_CUTS = sc.mean_transitions(a_tr, CONTROLS)
a_units = sc.to_segment_units(a_pos, AKI_CUTS)
GUARDS.append(f'G7: AKI control transitions ordered; cuts {AKI_CUTS[0]:.3f}, {AKI_CUTS[1]:.3f}')
a_pt = a_ident == 'Proximal_Tubule'
a_x = sparse.csr_matrix(AKI.X, dtype=float)[np.flatnonzero(a_mask)][a_pt]
a_genes_all = pd.Index(AKI.var_names.astype(str))
a_not_mt = ~a_genes_all.str.startswith('mt-')
a_lib = np.asarray(a_x[:, np.flatnonzero(a_not_mt)].sum(axis=1)).ravel()
a_spec, a_units = a_spec[a_pt], a_units[a_pt]
a_det = detection_by(a_x, a_spec, AKI4)
a_keep = (a_det >= DETECTION).all(axis=0) & a_not_mt
A_GENES = list(a_genes_all[a_keep])
a_x = a_x[:, np.flatnonzero(a_keep)]
A_USABLE = (a_det[:, a_keep] >= USABLE).all(axis=0)
A_INDEX = pd.Series(np.arange(len(A_GENES)), index=A_GENES)
a_bin, _ = pp.fixed_bins(a_units)
A_PROF = pp.bin_profiles(a_x, a_lib, a_bin, NB, a_spec, AKI4)
A_RATES = pp.bin_profiles(a_x, a_lib, a_bin, NB, a_spec, AKI4, centre=False)
a_valid = np.all([np.isfinite(A_PROF[n][0]) for n in AKI4], axis=0)
A_DEV = {n: A_PROF[n] - pp.reference_profile(A_PROF[n], A_USABLE) for n in AKI4}
print(f'AKI: {len(a_spec):,} U_PT units {pd.Series(a_spec).value_counts().to_dict()}; {len(A_GENES):,} genes '
      f'({int(A_USABLE.sum()):,} usable); valid bins {int(a_valid.sum())}/{NB}')


def a_delta(case, ref):
    d = np.mean([A_RATES[n] for n in case], axis=0) - np.mean([A_RATES[n] for n in ref], axis=0)
    return pp.centre_by_median(d, A_USABLE)


DELTA = {'condition': a_delta(IRS, CONTROLS),
         'relabel A2|A4': a_delta(['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4']),
         'relabel cross': a_delta(['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2'])}
PAIRS = [(i, c) for i in IRS for c in CONTROLS]
for i, c in PAIRS:
    DELTA[f'{i}-{c}'] = a_delta([i], [c])
SEG_MASK = {s: (CENTRES >= k) & (CENTRES < k + 1) & a_valid for k, s in enumerate(SEGMENTS)}

# coordination: 0.25-unit bins, depth-matched pseudobulks
c_bin, C_CENTRES = pp.fixed_bins(a_units, width=COORD_WIDTH)


def coordination_compute():
    rng = np.random.default_rng(SEED)
    out = {}
    for b in range(len(C_CENTRES)):
        libs = {n: a_lib[(a_spec == n) & (c_bin == b)].sum() for n in AKI4}
        target = min(libs.values())
        for n in AKI4:
            r = (a_spec == n) & (c_bin == b)
            if target <= 0:
                out[f'{n}|{b}'] = np.full(len(A_GENES), np.nan)
                continue
            counts = np.asarray(a_x[r].sum(axis=0)).ravel().astype(np.int64)
            out[f'{n}|{b}'] = np.log2((pp.thin_to(counts, libs[n], target, rng) + .5) / target * 1e6)
        out[f'target|{b}'] = np.array(target)
    return out


COORD = cached('coordination_pseudobulk', coordination_compute, params={'width': COORD_WIDTH, 'seed': SEED},
               inputs={'counts': digest(a_x), 'units': a_units, 'library': a_lib})
A_LOGMEAN = np.log2(np.mean([np.asarray(a_x[a_spec == n].sum(axis=0)).ravel() / a_lib[a_spec == n].sum() for n in AKI4], axis=0) * 1e4 + 1e-6)
A_DECILE = pd.qcut(A_LOGMEAN, 10, labels=False, duplicates='drop')


def coordination(idx):
    rows = []
    for b in range(len(C_CENTRES)):
        v = {n: COORD[f'{n}|{b}'][idx] for n in AKI4}
        ctrl = pp.pair_r2(v[CONTROLS[0]], v[CONTROLS[1]])
        ir = np.nanmean([pp.pair_r2(v[i], v[c]) for i, c in PAIRS])
        rows.append(ctrl - ir)
    return np.array(rows)


E4_ROWS, E4_DELTA, E4_COORD = [], {}, {}
rng_r = np.random.default_rng(SEED + 1)
for r in FEATURES.itertuples():
    genes = library_members(r.library_key, A_GENES)
    idx = A_INDEX.reindex(genes).dropna().astype(int).to_numpy()
    if len(idx) < MIN_FEATURE:
        E4_ROWS.append({'id': r.id, 'family': r.family, 'term': r.term, 'members': len(idx), 'class': 'too few members'})
        continue
    prof = {k: np.median(v[idx], axis=0) for k, v in DELTA.items()}
    E4_DELTA[r.id] = prof['condition']
    seg = {s: float(np.mean(prof['condition'][SEG_MASK[s]])) for s in SEGMENTS}
    order_s = sorted(SEGMENTS, key=lambda s: -abs(seg[s]))
    peak, second = order_s[0], order_s[1]
    margin = abs(seg[peak]) - abs(seg[second])
    sign = np.sign(seg[peak])
    pairs_agree = sum(np.sign(np.mean(prof[f'{i}-{c}'][SEG_MASK[peak]])) == sign for i, c in PAIRS)
    ratios = [abs(np.mean(prof[k][SEG_MASK[peak]])) / max(abs(seg[peak]), 1e-9) for k in ('relabel A2|A4', 'relabel cross')]
    present = [abs(seg[s]) >= EFFECT for s in SEGMENTS]
    if abs(seg[peak]) < EFFECT:
        cls = 'no change'
    elif margin >= EFFECT and pairs_agree == 4 and max(ratios) <= .5:
        cls = f'localized to {peak}'
    elif all(present) and (max(seg.values()) - min(seg.values())) < EFFECT and pairs_agree == 4 and max(ratios) <= .5:
        cls = 'uniform'
    else:
        cls = 'not resolved'
    dr2 = coordination(idx)
    null = np.array([coordination(np.array([rng_r.choice(np.flatnonzero(A_DECILE == A_DECILE[j])) for j in idx]))
                     for _ in range(N_RANDOM)])
    pct = (null < dr2).mean(axis=0)
    E4_COORD[r.id] = (dr2, pct)
    j = int(np.nanargmax(np.abs(np.where(a_valid, prof['condition'], np.nan))))
    E4_ROWS.append({'id': r.id, 'family': r.family, 'term': r.term, 'members': len(idx),
                    **{f'Δ {s}': seg[s] for s in SEGMENTS}, 'peak segment': peak, 'margin': margin,
                    'pairs agree': int(pairs_agree), 'relabeled ratio A2|A4': ratios[0], 'relabeled ratio cross': ratios[1],
                    'max |Δ| at': float(CENTRES[j]), 'class': cls,
                    'ΔR² S1': float(np.nanmean(dr2[C_CENTRES < 1])), 'ΔR² S2': float(np.nanmean(dr2[(C_CENTRES >= 1) & (C_CENTRES < 2)])),
                    'ΔR² S3': float(np.nanmean(dr2[C_CENTRES >= 2])),
                    'max ΔR² at': float(C_CENTRES[int(np.nanargmax(dr2))]),
                    'ΔR² percentile at max': float(pct[int(np.nanargmax(dr2))])})
E4 = pd.DataFrame(E4_ROWS)
E4.to_csv(tables / 'e4_disease_features.csv', index=False)
display(E4.round(3))
print('E4 classes:', E4['class'].value_counts().to_dict())

# E1 in the IR mice: each positional program on the AKI coordinate
E1_AKI = []
for k in order:
    reps_k = REP_TABLE[REP_TABLE.cluster.eq(k)]['set'].tolist()
    prof = {}
    for n in AKI4:
        ps = []
        for s in reps_k:
            idx = A_INDEX.reindex([m['genes'][i] for i in SET_MEMBERS[s]]).dropna().astype(int).to_numpy()
            idx = idx[A_USABLE[idx]]
            if len(idx) >= MIN_SET:
                ps.append(np.median(A_DEV[n][idx], axis=0))
        prof[n] = np.mean(ps, axis=0) if ps else np.full(NB, np.nan)
    ctrl = (prof[CONTROLS[0]] + prof[CONTROLS[1]]) / 2
    for n in IRS:
        ok = np.isfinite(prof[n]) & np.isfinite(ctrl)
        E1_AKI.append({'program': PROGRAM_NAME[k], 'IR mouse': n, 'r with control': float(np.corrcoef(prof[n][ok], ctrl[ok])[0, 1]),
                       'amplitude ratio': float(np.ptp(prof[n][ok]) / np.ptp(ctrl[ok]))})
    E1_AKI.append({'program': PROGRAM_NAME[k], 'IR mouse': 'controls (A2 vs A4)',
                   'r with control': float(np.corrcoef(prof[CONTROLS[0]][a_valid], prof[CONTROLS[1]][a_valid])[0, 1]),
                   'amplitude ratio': float(np.ptp(prof[CONTROLS[0]][a_valid]) / np.ptp(prof[CONTROLS[1]][a_valid]))})
    E4_DELTA[f'program {PROGRAM_NAME[k]}'] = prof
E1_AKI = pd.DataFrame(E1_AKI)
E1_AKI.to_csv(tables / 'e1_programs_in_ir_mice.csv', index=False)
display(E1_AKI.round(3))

## 9b · Post hoc (descriptive; outside the protocol)

Added after the run, to read the frozen results: (1) single genes with an agreed boundary peak or trough, against a
null that shuffles each mouse's bin order; (2) the sets closest to the boundary rule; (3) why the cluster view at
0.25 calls almost none of the eligible sets, and what Chen's microdissection calls for them; (4) coordination by
segment.

In [ ]:
GENE_DEV = {n: m['deviation'][n][U_IDX] for n in MOUSE}
gcalls = boundary_calls(GENE_DEV, CENTRES)
G_OBS = {c: int((gcalls[c] >= EFFECT).sum()) for c in ('E1', 'T1', 'E2', 'T2')}
rng_g = np.random.default_rng(SEED + 2)
G_NULL = {c: [] for c in G_OBS}
for _ in range(200):
    shuffled = {n: GENE_DEV[n][:, rng_g.permutation(NB)] for n in MOUSE}
    gc = boundary_calls(shuffled, CENTRES)
    for c in G_OBS:
        G_NULL[c].append(int((gc[c] >= EFFECT).sum()))
GENE_BOUNDARY = pd.DataFrame([{'call': c, 'genes': G_OBS[c], 'null mean': float(np.mean(G_NULL[c])),
                               'null 95th': float(np.quantile(G_NULL[c], .95)),
                               'p': (1 + sum(v >= G_OBS[c] for v in G_NULL[c])) / 201} for c in G_OBS])
gcalls.index = [m['genes'][i] for i in U_IDX]
gene_rows = []
for c in ('E1', 'T1', 'E2', 'T2'):
    top = gcalls[gcalls[c] >= EFFECT].sort_values(c, ascending=False)
    for g, r in top.iterrows():
        gene_rows.append({'gene': g, 'call': c, 'excess (agreed)': float(r[c]), 'log2 CP10k': float(LOG_CP[m['index'][g]])})
GENE_LIST = pd.DataFrame(gene_rows)
GENE_BOUNDARY.to_csv(tables / 'posthoc_boundary_genes_summary.csv', index=False)
GENE_LIST.to_csv(tables / 'posthoc_boundary_genes.csv', index=False)
display(GENE_BOUNDARY)
display(GENE_LIST.groupby('call').head(12))

NEAR = CALLS.assign(name=[short(s) for s in CALLS.index]).sort_values('agreed excess', ascending=False).head(12)
NEAR[['name', 'agreed excess', 'E1', 'T1', 'E2', 'T2']].to_csv(tables / 'posthoc_near_boundary_sets.csv')
display(NEAR[['name', 'agreed excess', 'E1', 'T1', 'E2', 'T2']].round(3))

cv_max = np.array([np.abs(cluster_view([m['genes'][j] for j in SET_MEMBERS[s]])[0].mean(axis=0)).max() for s in ELIG_IDS])
curve_amp = np.array([np.ptp(MEAN_PROF[SET_IDS.index(s)]) for s in ELIG_IDS])
E2['curve amplitude'], E2['cluster max |value|'] = curve_amp, cv_max
E2['Chen'] = np.where(E2['Chen call'].eq('none'), 'not called', 'called')
E2.to_csv(tables / 'e2_cluster_vs_curve.csv', index=False)
print(f'E2 post hoc: eligible sets have curve amplitude median {np.median(curve_amp):.2f} log2 '
      f'(IQR {np.quantile(curve_amp, .25):.2f}–{np.quantile(curve_amp, .75):.2f}); the cluster view\'s largest '
      f'|segment value| median {np.median(cv_max):.2f} ({(cv_max >= EFFECT).mean():.2f} reach {EFFECT}). '
      f'Chen calls {int((E2.Chen == "called").sum())} of {len(E2)} eligible sets.')
display(pd.crosstab(REP_TABLE.program.replace('', 'unassigned'), REP_TABLE['Chen call']))

COORD_LONG = pd.DataFrame([{'id': f, 'bin centre': C_CENTRES[b], 'ΔR²': v[0][b], 'percentile': v[1][b]}
                           for f, v in E4_COORD.items() for b in range(len(C_CENTRES))])
COORD_LONG['segment'] = pd.cut(COORD_LONG['bin centre'], [0, 1, 2, 2.5], labels=SEGMENTS)
COORD_SEG = COORD_LONG.groupby('segment', observed=True).agg(median_dR2=('ΔR²', 'median'),
                                                             share_pct90=('percentile', lambda p: (p >= .9).mean()))
COORD_LONG.to_csv(tables / 'e4_coordination_by_bin.csv', index=False)
display(COORD_SEG.round(3))

## 10 · Figures

In [ ]:
def seg_lines(ax, cuts=(1, 2)):
    for b in cuts:
        ax.axvline(b, color=LIGHT, lw=.6, zorder=0)


# 89a · T
fig, axes = plt.subplots(2, 4, figsize=(180 * MM, 82 * MM), gridspec_kw={'width_ratios': [1.2, .7, 1, 1]})
ex = meta[(meta['shape'] == 'bump c=1.0 w=0.5') & meta.A.eq(1.) & meta.arm.eq('realistic')].index
ex_s = meta[(meta['shape'] == 'step-matched c=1.0 w=0.5') & meta.A.eq(1.) & meta.arm.eq('realistic')].index
for row, (rows_b, rows_s) in enumerate([(ex, ex_s), (meta[(meta['shape'] == 'bump c=2.0 w=0.5') & meta.A.eq(1.) & meta.arm.eq('realistic')].index,
                                                     meta[(meta['shape'] == 'step-matched c=2.0 w=0.5') & meta.A.eq(1.) & meta.arm.eq('realistic')].index)]):
    ax = axes[row, 0]
    for rr, col in ((rows_b, '#C0392B'), (rows_s, GREY)):
        y = mean_prof[rr]
        ax.fill_between(CENTRES, *np.quantile(y, [.1, .9], axis=0), color=col, alpha=.2, lw=0)
        ax.plot(CENTRES, np.median(y, axis=0), color=col)
    seg_lines(ax)
    ax.set_ylabel('log2, centred')
    ax = axes[row, 1]
    for j, (rr, col) in enumerate(((rows_b, '#C0392B'), (rows_s, GREY))):
        v = np.mean([SIM_LAB[n][rr] for n in MOUSE], axis=0)
        ax.bar(np.arange(3) + (j - .5) * .38, np.median(v, axis=0), width=.36, color=col,
               yerr=np.abs(np.quantile(v, [.1, .9], axis=0) - np.median(v, axis=0)), error_kw={'lw': .5})
    ax.set_xticks(range(3), SEGMENTS)
    ax.axhline(0, color=INK, lw=.4)
for j, arm in enumerate(('ideal', 'realistic')):
    ax = axes[0, 2 + j]
    sub = T_SUMMARY[T_SUMMARY.kind.eq('bump') & T_SUMMARY.c.eq(1.) & T_SUMMARY.arm.eq(arm)].pivot(index='w', columns='A', values='own_boundary')
    ax.imshow(sub.to_numpy(), cmap='Greys', vmin=0, vmax=1, origin='lower', aspect='auto')
    for (a_, b_), v in np.ndenumerate(sub.to_numpy()):
        ax.text(b_, a_, f'{v:.2f}', ha='center', va='center', fontsize=5.5, color='white' if v > .5 else INK)
    ax.set_xticks(range(sub.shape[1]), sub.columns)
    ax.set_yticks(range(sub.shape[0]), sub.index)
    ax.set_xlabel('A (log2)')
    ax.set_ylabel('w (segment units)')
ax = axes[1, 2]
rr = meta[meta.kind.eq('ramp') & meta.c.eq(1.) & meta.A.eq(1.) & meta.arm.eq('realistic')]
for k, w in enumerate((.1, .5, 1.5)):
    v = rr[rr.w.eq(w)]['edge width'].dropna()
    ax.scatter(np.full(len(v), k) + np.random.default_rng(k).uniform(-.15, .15, len(v)), v, s=2, color=INK, alpha=.4, lw=0)
    ax.plot([k - .25, k + .25], [w, w], color='#C0392B')
ax.set_yscale('log')
ax.set_xticks(range(3), ['0.1', '0.5', '1.5'])
ax.set_xlabel('true width')
ax.set_ylabel('recovered width')
ax = axes[1, 3]
ll = meta[meta.kind.eq('local') & meta.A.eq(1.) & meta.arm.eq('realistic')]
for k, c in enumerate((.3, .7, 1.3, 1.7)):
    v = ll[ll.c.eq(c)]['peak position']
    ax.scatter(np.full(len(v), c) + np.random.default_rng(k).uniform(-.04, .04, len(v)), v, s=2, color=INK, alpha=.4, lw=0)
ax.plot([0, 2], [0, 2], color='#C0392B', lw=.6)
seg_lines(ax, (1,))
ax.set_xlabel('true position')
ax.set_ylabel('recovered position')
fig.tight_layout()
save(fig, 'fig89a_showable')

# 89b · E1 positional programs
rep_order = [i for k in order for i in np.flatnonzero(labels == k)]
fig, axes = plt.subplots(1, 3, figsize=(180 * MM, 95 * MM), gridspec_kw={'width_ratios': [1.4, 1, 1]})
ax = axes[0]
M = MEAN_PROF[ri][rep_order]
im = ax.imshow(M, cmap='RdBu_r', norm=TwoSlopeNorm(0, -1, 1), aspect='auto', extent=(0, 2.5, len(M) - .5, -.5), interpolation='nearest')
fig.colorbar(im, ax=ax, shrink=.4, label='log2, centred')
y0 = 0
for k in order:
    n_k = int((labels == k).sum())
    ax.axhline(y0 + n_k - .5, color=INK, lw=.5)
    y0 += n_k
seg_lines(ax)
ax.set_yticks([])
ax.set_xlabel('mouse segment units')
ax = axes[1]
for j, k in enumerate(order):
    off = -j * 1.2
    for n in MOUSE:
        ax.plot(CENTRES, OBS_PROF[n][ri][labels == k].mean(axis=0) + off, color=SPEC_COLOR[n])
    ax.axhline(off, color=LIGHT, lw=.4)
seg_lines(ax)
ax.set_yticks([-j * 1.2 for j in range(len(order))], [PROGRAM_NAME[k] for k in order])
ax.set_xlabel('mouse segment units')
ax = axes[2]
for j, k in enumerate(order):
    off = -j * 1.2
    for n in HUMAN:
        ps = []
        for s in REP_TABLE[REP_TABLE.cluster.eq(k)]['set']:
            hm = members_of(human_symbols([m['genes'][i] for i in SET_MEMBERS[s]]), 'human')
            if len(hm) >= MIN_SET:
                ps.append(np.median(hh['deviation'][n][hm], axis=0))
        if ps:
            ax.plot(H_CENTRES[n], np.mean(ps, axis=0) + off, color=SPEC_COLOR[n])
    ax.axhline(off, color=LIGHT, lw=.4)
for n in HUMAN:
    for t in HUMAN_CUTS[n]:
        ax.axvline(t, color=SPEC_COLOR[n], lw=.4, ls=':', zorder=0)
ax.set_yticks([-j * 1.2 for j in range(len(order))], [PROGRAM_NAME[k] for k in order])
ax.set_xlabel('human curve position')
fig.tight_layout()
save(fig, 'fig89b_positional_programs')

# 89c · H1: set-level count against its null; single genes against the bin-shuffle null (post hoc); the sets closest
# to the rule (post hoc)
fig, axes = plt.subplots(2, 4, figsize=(180 * MM, 80 * MM))
ax = axes[0, 0]
ax.hist(H1['null_count'], bins=np.arange(max(H1['null_count'].max(), C_OBS) + 2) - .5, color=GREY)
ax.axvline(C_OBS, color='#C0392B')
ax.set_xlabel('boundary programs')
ax.set_ylabel('permutations')
for a, c in zip(axes[0, 1:], ('E1', 'T1', 'E2')):
    a.hist(G_NULL[c], bins=20, color=GREY)
    a.axvline(G_OBS[c], color='#C0392B')
    a.set_xlabel(f'genes ({c})')
show = BOUNDARY_TABLE.head(4)['set'].tolist() if len(BOUNDARY_TABLE) else NEAR.index[:4].tolist()
for a, s_id in zip(axes[1], show):
    i = SET_IDS.index(s_id)
    for n in MOUSE:
        a.plot(CENTRES, OBS_PROF[n][i], color=SPEC_COLOR[n])
    vals, _ = cluster_view([m['genes'][j] for j in SET_MEMBERS[s_id]])
    for n_i, n in enumerate(MOUSE):
        a.hlines(vals[n_i], [0, 1, 2], [1, 2, 2.5], color=SPEC_COLOR[n], lw=2.2, alpha=.35)
    seg_lines(a)
    a.axhline(0, color=LIGHT, lw=.4)
    a.set_title(short(s_id)[:38], fontsize=5.5)
fig.tight_layout()
save(fig, 'fig89c_boundary_programs')

# 89d · E2
fig, axes = plt.subplots(1, 4, figsize=(180 * MM, 45 * MM), gridspec_kw={'width_ratios': [.8, 1, 1, 1]})
ax = axes[0]
ax.bar(range(4), E2_COUNTS.to_numpy(), color=[GREY, '#C0392B', '#E69F00', INK])
ax.set_xticks(range(4), E2_COUNTS.index, rotation=45, ha='right')
ax.set_ylabel('eligible sets')
for a, cat in zip(axes[1:], ['boundary', 'mislocated', 'missed']):
    sub = E2[E2.category.eq(cat)]
    if len(sub):
        s = sub.iloc[np.argmax([np.ptp(MEAN_PROF[SET_IDS.index(x)]) for x in sub['set']])]['set']
        i = SET_IDS.index(s)
        for n in MOUSE:
            a.plot(CENTRES, OBS_PROF[n][i], color=SPEC_COLOR[n])
        vals, _ = cluster_view([m['genes'][j] for j in SET_MEMBERS[s]])
        for n_i, n in enumerate(MOUSE):
            a.hlines(vals[n_i], [0, 1, 2], [1, 2, 2.5], color=SPEC_COLOR[n], lw=2.2, alpha=.35)
        a.set_title(short(s)[:38], fontsize=5.5)
        seg_lines(a)
        a.axhline(0, color=LIGHT, lw=.4)
    else:
        a.axis('off')
fig.tight_layout()
save(fig, 'fig89d_cluster_vs_curve')

# 89e · E3
e3_ids = [p for p in E3_PROF if not p.startswith(('P', 'K', 'C', 'H', 'R', 'N'))] + [p for p in E3_PROF if p.startswith(('N', 'C', 'H', 'R', 'P', 'K'))]
fig, ax = plt.subplots(figsize=(110 * MM, max(60, 2.6 * len(e3_ids)) * MM))
M = np.array([(E3_PROF[p][MOUSE[0]] + E3_PROF[p][MOUSE[1]]) / 2 for p in e3_ids])
im = ax.imshow(M, cmap='RdBu_r', norm=TwoSlopeNorm(0, -1, 1), aspect='auto', extent=(0, 2.5, len(M) - .5, -.5), interpolation='nearest')
fig.colorbar(im, ax=ax, shrink=.3, label='log2, centred')
ax.set_yticks(range(len(e3_ids)), e3_ids, fontsize=5)
seg_lines(ax)
ax.set_xlabel('mouse segment units')
fig.tight_layout()
save(fig, 'fig89e_cellspectra_programs')

# 89f · E4
e4_ids = [r for r in E4.id if r in E4_DELTA]
fig, axes = plt.subplots(1, 3, figsize=(180 * MM, 95 * MM), gridspec_kw={'width_ratios': [1.2, 1, 1]})
ax = axes[0]
M = np.array([np.where(a_valid, E4_DELTA[r], np.nan) for r in e4_ids])
im = ax.imshow(M, cmap='RdBu_r', norm=TwoSlopeNorm(0, -1, 1), aspect='auto', extent=(0, 2.5, len(M) - .5, -.5), interpolation='nearest')
fig.colorbar(im, ax=ax, shrink=.3, label='IR − control (log2)')
ax.set_yticks(range(len(e4_ids)), e4_ids, fontsize=5)
seg_lines(ax)
ax.set_xlabel('mouse segment units')
ax = axes[1]
M = np.array([E4_COORD[r][0] for r in e4_ids])
im = ax.imshow(M, cmap='PuOr_r', norm=TwoSlopeNorm(0, -.5, .5), aspect='auto', extent=(0, 2.5, len(M) - .5, -.5), interpolation='nearest')
fig.colorbar(im, ax=ax, shrink=.3, label='ΔR² (control − IR)')
ax.set_yticks(range(len(e4_ids)), e4_ids, fontsize=5)
seg_lines(ax)
ax.set_xlabel('mouse segment units')
ax = axes[2]
for j, k in enumerate(order):
    off = -j * 1.2
    prof = E4_DELTA[f'program {PROGRAM_NAME[k]}']
    for n in AKI4:
        ax.plot(CENTRES, np.where(a_valid, prof[n], np.nan) + off, color=SPEC_COLOR[n], lw=.8)
    ax.axhline(off, color=LIGHT, lw=.4)
seg_lines(ax)
ax.set_yticks([-j * 1.2 for j in range(len(order))], [PROGRAM_NAME[k] for k in order])
ax.set_xlabel('mouse segment units')
fig.tight_layout()
save(fig, 'fig89f_disease')

## 11 · Readout and run record

In [ ]:
READOUT = pd.DataFrame([
    {'part': 'T', 'readout': 'showable (realistic, A = 1)', 'value': json.dumps({k: bool(v) for k, v in SHOWABLE.items()})},
    {'part': 'H1', 'readout': 'boundary programs', 'value': f'{C_OBS} ({N_GROUPS_B} groups); null mean '
                                                            f'{H1["null_count"].mean():.2f}; p {P_H1:.4f}; holds {H1_HOLDS}'},
    {'part': 'E1', 'readout': 'positional programs', 'value': f'{len(programs)} programs, K {K}, ARI {ARI:.3f} (K 6: {ARI6:.3f})'},
    {'part': 'E2', 'readout': 'cluster view vs curve', 'value': json.dumps({k: int(v) for k, v in E2_COUNTS.items()})},
    {'part': 'E4', 'readout': 'disease classes', 'value': json.dumps(E4['class'].value_counts().to_dict())}])
READOUT.to_csv(tables / 'readout.csv', index=False)
display(READOUT)
record = {'logic': NOTEBOOK_LOGIC_VERSION, 'guards': GUARDS, 'reading': READING, 'seconds': SECONDS,
          'external_sha256': EXTERNAL_SHA, 'units': {n: int((U_PT & (SPECIMEN == n)).sum()) for n in MOUSE + HUMAN},
          'aki_units': pd.Series(a_spec).value_counts().to_dict(), 'mouse_cuts': list(MOUSE_CUTS), 'aki_cuts': list(AKI_CUTS),
          'refit_coverage': cover, 'sets_tested': len(SET_IDS), 'h1': {'C': C_OBS, 'groups': N_GROUPS_B, 'p': P_H1},
          'e1': {'eligible': int(ELIG.sum()), 'representatives': len(REPS), 'K': int(K), 'silhouette': SIL, 'ARI': ARI, 'ARI_K6': ARI6},
          'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=1, default=float))
print(f'Done in {time.perf_counter() - NOTEBOOK_START:.0f} s')